# IMU Segraw → Moments Feature Engineering

Pipeline:
1. Load raw segmented IMU CSVs into nested dict
2. Skip BPF (IMU is low-frequency; BPF at 20-450 Hz would destroy it)
3. Per-gesture std normalisation (same as EMG pipeline)
4. Khushaba spectral moments FE (5 features × 90 IMU channels = 450 features per gesture)
5. Global feature-matrix std normalisation
6. Save to CSV — one row per gesture trial

Mirror of `ppdsegraw_feature_engineering.ipynb` for EMG, with two changes:
- `modalities=['I']` in `load_segraw_data`
- `already_BPFd=True` in `apply_filter_to_nested_dict` (skips BPF)

Output file: `moments_segraw_allIMU.csv`

In [1]:
import numpy as np
import pandas as pd
import time
import json

# All shared preprocessing + FE functions live here
from shared_processing import (
    load_segraw_data,
    apply_filter_to_nested_dict,
    normalize_gestures_by_std_any_channels,
    create_khushaba_spectralmomentsFE_vectors,
    normalize_whole_dataset_features,
)

## 0. Config — set your paths here

In [ ]:
# ── UPDATE THESE TO YOUR MACHINE ──────────────────────────────────────────
# C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data
RAW_DATA_DIR   = r"C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data"
SAVE_DIR       = r"C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project"
# ──────────────────────────────────────────────────────────────────────────

# Intermediate save (nested dict after loading, before FE) — optional but
# saves ~5 min if you need to re-run FE without reloading CSVs
RAW_DICT_SAVE_PATH      = SAVE_DIR + r"\segraw0_IMU_allgestures_allusers.json"
PPD_DICT_SAVE_PATH      = SAVE_DIR + r"\ppdsegraw1_allIMU.json"
MOMENTS_CSV_SAVE_PATH   = SAVE_DIR + r"\moments_ppdsegraw2_allIMU.csv"

# Participant lists — identical to your EMG pipeline
pIDs_impaired   = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
                   'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124',
                   'P125','P126','P127','P128','P131','P132']
pIDs_unimpaired = ['P004','P005','P006','P008','P010','P011']
pIDs_both       = pIDs_impaired + pIDs_unimpaired

# NOTE: 2000 Hz was for EMG
# 148 Hz was for IMU
FS = 148  # Nominal sampling rate stored in the raw files (used for moment calculations)

## 1. Load raw IMU CSVs

> Same `load_segraw_data` as the EMG pipeline — just pass `modalities=['I']`.
> Each gesture ends up as a list of 72 channels × ~300 timepoints.

In [3]:
# This took 50 minutes for exp-def IMU

print("Loading raw IMU data...")
start = time.time()

nested_dict = load_segraw_data(
    pIDs      = pIDs_both,
    data_dir_path = RAW_DATA_DIR,
    modalities    = ["I"],          # ← IMU only (was ["E"] for EMG pipeline)
    expt_types    = ["standardized"],
    num_imu_sensors = 12,
)

print(f"Done in {time.time()-start:.1f}s")
print(f"Participants loaded: {list(nested_dict.keys())}")

Loading raw IMU data...
P102
P103
P104
P105
P106
P107
P108
P109
P110
P111
P112
P114
P115
P116
P118
P119
P121
P122
P123
P124
P125
P126
P127
P128
P131
P132
P004
P005
P006
P008
P010
P011
Done in 986.3s
Participants loaded: ['P104', 'P105', 'P111', 'P114', 'P116', 'P119', 'P123', 'P124', 'P126', 'P131', 'P132', 'P004', 'P005', 'P006', 'P008', 'P010', 'P011']


In [4]:
# Quick sanity check — should be 72 channels (12 sensors × 6 axes) --> It might think there's 15 sensors since the highest sensor channel was labeled 15...
sample_pid     = list(nested_dict.keys())[0]
sample_gesture = list(nested_dict[sample_pid].keys())[0]
sample_trial   = list(nested_dict[sample_pid][sample_gesture].keys())[0]
sample_data    = nested_dict[sample_pid][sample_gesture][sample_trial]["IMU"]

print(f"Sample: {sample_pid} / {sample_gesture} / trial {sample_trial}")
print(f"  n_channels  : {len(sample_data)}   (expected 72)")
print(f"  n_timepoints: {len(sample_data[0])}  (expected ~300)")

Sample: P104 / two-handed-tap / trial 1
  n_channels  : 72   (expected 72)
  n_timepoints: 999  (expected ~300)


In [5]:
# Took 5 minutes to save lol

# Optional: save the raw nested dict so you don't have to reload CSVs next time
print("Saving raw IMU nested dict...")
import json
with open(RAW_DICT_SAVE_PATH, 'w') as f:
    json.dump(nested_dict, f)
print(f"Saved → {RAW_DICT_SAVE_PATH}")

Saving raw IMU nested dict...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segraw_IMU_allgestures_allusers.json


## 2. Normalise — skip BPF, mean-subtract, then per-gesture std=1

> **Key difference from EMG pipeline:** `already_BPFd=True` skips the
> 20-450 Hz bandpass filter entirely.  IMU signals are low-frequency;
> that BPF would zero out almost everything.
>
> We still do mean subtraction + per-gesture std normalisation,
> which is the same as the EMG pipeline.

In [6]:
print("Mean-subtracting (no BPF)...")
start = time.time()

ms_dict = apply_filter_to_nested_dict(
    nested_dict,
    normalization_method = "MEANSUBTRACTION",
    already_BPFd         = True,   # ← skips BPF
)

print(f"Done in {time.time()-start:.1f}s")

Mean-subtracting (no BPF)...
Done in 3.0s


In [7]:
print("Per-gesture std normalisation...")
start = time.time()

# Use the any-channels variant so it handles 90 IMU channels automatically
ppd_dict = normalize_gestures_by_std_any_channels(ms_dict)

print(f"Done in {time.time()-start:.1f}s")

# Sanity check: std across flattened channels should be ~1
sample_data_ppd = ppd_dict[sample_pid][sample_gesture][sample_trial]["IMU"]
flat = [v for ch in sample_data_ppd for v in ch]
print(f"  Post-normalisation std (should be ~1.0): {np.std(flat):.4f}")

Per-gesture std normalisation...
Done in 3.5s
  Post-normalisation std (should be ~1.0): 1.0000


In [8]:
# Optional: save the preprocessed dict
print("Saving preprocessed IMU dict...")
with open(PPD_DICT_SAVE_PATH, 'w') as f:
    json.dump(ppd_dict, f)
print(f"Saved → {PPD_DICT_SAVE_PATH}")

Saving preprocessed IMU dict...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\ppdsegraw_allIMU.json


## 3. Khushaba Moments Feature Engineering

For each gesture trial, for each of the 90 IMU channels, we compute:
- `zero_order`        — total signal power (energy)
- `second_order`      — energy of rate-of-change (first derivative)
- `fourth_order`      — energy of jerk (second derivative)
- `sparsity`          — sqrt(m0*m4) / m2
- `irregularity_factor` — m2^2 / (m0*m4), normalised by waveform length

Result: **360 features per gesture trial** (72 channels × 5 features).
All degenerate values (e.g. near-constant channels) are handled by
`safe_log` — clipped to [-10, 10] rather than blowing up to ±inf.

In [9]:
print("Computing Khushaba moments...")
start = time.time()

moments_df = create_khushaba_spectralmomentsFE_vectors(ppd_dict, fs=FS)

print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {moments_df.shape}   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])")
moments_df.head()

Computing Khushaba moments...
Done in 14.8s
Shape: (1115, 363)   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])


,Participant,Gesture_ID,Gesture_Num,IMU0_zero_order,IMU0_second_order,IMU0_fourth_order,IMU0_sparsity,IMU0_irregularity_factor,IMU1_zero_order,IMU1_second_order,...,IMU70_zero_order,IMU70_second_order,IMU70_fourth_order,IMU70_sparsity,IMU70_irregularity_factor,IMU71_zero_order,IMU71_second_order,IMU71_fourth_order,IMU71_sparsity,IMU71_irregularity_factor
0,P104,two-handed-tap,1,-1.864447,-1.926367,0.322930,1.155609,1.166657,-2.814384,-0.517795,...,4.511053,-10.0,-10.0,1.983958,-1.469556,1.942617,-7.715451,-10.0,1.674185,-0.087541
1,P104,two-handed-tap,2,-2.704036,-0.183122,3.507391,0.584802,1.878935,-3.515966,0.702581,...,3.848287,-10.0,-10.0,2.116095,-1.178987,2.359543,-8.648829,-10.0,1.640834,-0.038066
2,P104,two-handed-tap,3,-2.562236,-0.825655,2.856423,0.972749,1.597615,-2.686883,-1.399570,...,4.218129,-10.0,-10.0,2.295843,-1.232377,1.806791,-7.963350,-10.0,1.865712,0.103230
3,P104,two-handed-tap,4,-2.110510,-0.676876,1.731346,0.487294,1.979558,-3.168804,-0.225039,...,3.983050,-10.0,-10.0,2.095157,-0.999318,1.346591,-7.451094,-10.0,1.867306,0.324918
4,P104,two-handed-tap,5,-2.832553,-0.654716,3.611098,1.043993,1.731105,-3.376213,-0.139339,...,3.980252,-10.0,-10.0,2.185088,-1.170859,2.016673,-8.625173,-10.0,1.896105,0.166089


In [10]:
# Check for any surviving NaN or inf (should be zero with safe_log)
n_nan = moments_df.isna().sum().sum()
n_inf = np.isinf(moments_df.select_dtypes(include=np.number).values).sum()
print(f"NaN count: {n_nan}  |  Inf count: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("WARNING: Unexpected NaN/inf — imputing with column mean as fallback.")
    moments_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    feature_cols = [c for c in moments_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
    moments_df[feature_cols] = moments_df[feature_cols].fillna(moments_df[feature_cols].mean())
else:
    print("Clean — no NaN or inf values.")

NaN count: 0  |  Inf count: 0
Clean — no NaN or inf values.


## 4. Global feature-matrix normalisation

Divides every feature by the global standard deviation of the entire
feature matrix, so all 450 features are on the same scale.
This is the same final step as in the EMG pipeline.

In [11]:
n_moments_df = normalize_whole_dataset_features(moments_df)

print(f"Shape after normalisation: {n_moments_df.shape}")
n_moments_df.head()

Shape after normalisation: (1115, 363)


,Participant,Gesture_ID,Gesture_Num,IMU0_zero_order,IMU0_second_order,IMU0_fourth_order,IMU0_sparsity,IMU0_irregularity_factor,IMU1_zero_order,IMU1_second_order,...,IMU70_zero_order,IMU70_second_order,IMU70_fourth_order,IMU70_sparsity,IMU70_irregularity_factor,IMU71_zero_order,IMU71_second_order,IMU71_fourth_order,IMU71_sparsity,IMU71_irregularity_factor
0,P104,two-handed-tap,1,-0.416984,-0.430832,0.072223,0.258452,0.260923,-0.629438,-0.115805,...,1.008898,-2.236502,-2.236502,0.443713,-0.328667,0.434467,-1.725562,-2.236502,0.374432,-0.019578
1,P104,two-handed-tap,2,-0.604758,-0.040955,0.784429,0.130791,0.420224,-0.786346,0.157132,...,0.860670,-2.236502,-2.236502,0.473265,-0.263681,0.527712,-1.934313,-2.236502,0.366973,-0.008514
2,P104,two-handed-tap,3,-0.573045,-0.184658,0.638840,0.217556,0.357307,-0.600922,-0.313014,...,0.943385,-2.236502,-2.236502,0.513466,-0.275621,0.404089,-1.781005,-2.236502,0.417267,0.023087
3,P104,two-handed-tap,4,-0.472016,-0.151384,0.387216,0.108984,0.442729,-0.708704,-0.050330,...,0.890810,-2.236502,-2.236502,0.468582,-0.223498,0.301165,-1.666439,-2.236502,0.417623,0.072668
4,P104,two-handed-tap,5,-0.633501,-0.146427,0.807623,0.233489,0.387162,-0.755091,-0.031163,...,0.890184,-2.236502,-2.236502,0.488695,-0.261863,0.451029,-1.929022,-2.236502,0.424064,0.037146


## 5. Save

In [12]:
n_moments_df.to_csv(MOMENTS_CSV_SAVE_PATH, index=False, header=True)
print(f"Saved → {MOMENTS_CSV_SAVE_PATH}")
print(f"Final shape: {n_moments_df.shape}")
print(f"Columns: {list(n_moments_df.columns[:6])} ... {list(n_moments_df.columns[-3:])}")

Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\moments_segraw_allIMU.csv
Final shape: (1115, 363)
Columns: ['Participant', 'Gesture_ID', 'Gesture_Num', 'IMU0_zero_order', 'IMU0_second_order', 'IMU0_fourth_order'] ... ['IMU71_fourth_order', 'IMU71_sparsity', 'IMU71_irregularity_factor']


## 6. Quick verification

In [13]:
# Reload and spot-check
check_df = pd.read_csv(MOMENTS_CSV_SAVE_PATH)
print(f"Reloaded shape : {check_df.shape}")
print(f"Participants   : {sorted(check_df['Participant'].unique())}")
print(f"Gestures       : {sorted(check_df['Gesture_ID'].unique())}")
print(f"Trials per gesture per participant (should be 10):")
print(check_df.groupby(['Participant','Gesture_ID'])['Gesture_Num'].count().describe())

feature_cols = [c for c in check_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
print(f"\nFeature stats (post-normalisation):")
print(check_df[feature_cols].describe().loc[['mean','std','min','max']])

Reloaded shape : (1115, 363)
Participants   : ['P004', 'P005', 'P006', 'P008', 'P010', 'P011', 'P104', 'P105', 'P111', 'P114', 'P116', 'P119', 'P123', 'P124', 'P126', 'P131', 'P132']
Gestures       : ['air-tap', 'double-clench', 'double-pinch', 'palm-pinch', 'pinch-and-scroll', 'point-and-pinch', 'shake-and-release', 'single-clench', 'single-pinch', 'two-handed-tap']
Trials per gesture per participant (should be 10):
count    115.000000
mean       9.695652
std        1.433765
min        3.000000
25%       10.000000
50%       10.000000
75%       10.000000
max       10.000000
Name: Gesture_Num, dtype: float64

Feature stats (post-normalisation):
      IMU0_zero_order  IMU0_second_order  IMU0_fourth_order  IMU0_sparsity   
mean        -0.849898           0.159258           1.222791       0.192101  \
std          0.457165           0.599676           1.150754       0.101676   
min         -1.768280          -1.763774          -2.236502       0.036600   
max          0.368852           1.34